In [2]:
import mlflow
import pandas as pd
import mlflow.sklearn
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
import pandas as pd
import re
import string
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
import numpy as np

In [3]:
df = pd.read_csv('IMDB.csv')
df = df.sample(500)
df.to_csv('data.csv', index=False)
df.head()

,review,sentiment
489,The main reason to see this film is Warren Wil...,positive
255,"Well, not much really to say about this. But i...",positive
52,This is an EXCELLENT example of early Bette Da...,positive
129,"An art house maven's dream. Overrated, overpra...",negative
818,"It's a poor film, but I must give it to the le...",negative


In [4]:
# data preprocessing

# Define text preprocessing functions
def lemmatization(text):
    """Lemmatize the text."""
    lemmatizer = WordNetLemmatizer()
    text = text.split()
    text = [lemmatizer.lemmatize(word) for word in text]
    return " ".join(text)

def remove_stop_words(text):
    """Remove stop words from the text."""
    stop_words = set(stopwords.words("english"))
    text = [word for word in str(text).split() if word not in stop_words]
    return " ".join(text)

def removing_numbers(text):
    """Remove numbers from the text."""
    text = ''.join([char for char in text if not char.isdigit()])
    return text

def lower_case(text):
    """Convert text to lower case."""
    text = text.split()
    text = [word.lower() for word in text]
    return " ".join(text)

def removing_punctuations(text):
    """Remove punctuations from the text."""
    text = re.sub('[%s]' % re.escape(string.punctuation), ' ', text)
    text = text.replace('؛', "")
    text = re.sub('\s+', ' ', text).strip()
    return text

def removing_urls(text):
    """Remove URLs from the text."""
    url_pattern = re.compile(r'https?://\S+|www\.\S+')
    return url_pattern.sub(r'', text)

def normalize_text(df):
    """Normalize the text data."""
    try:
        df['review'] = df['review'].apply(lower_case)
        df['review'] = df['review'].apply(remove_stop_words)
        df['review'] = df['review'].apply(removing_numbers)
        df['review'] = df['review'].apply(removing_punctuations)
        df['review'] = df['review'].apply(removing_urls)
        df['review'] = df['review'].apply(lemmatization)
        return df
    except Exception as e:
        print(f'Error during text normalization: {e}')
        raise

In [7]:
df = normalize_text(df)
df.head()

,review,sentiment
489,main reason see film warren william top form s...,positive
255,well much really say really good good job soft...,positive
52,excellent example early bette davis talent pro...,positive
129,art house maven dream overrated overpraised ov...,negative
818,poor film must give lead actress one francine ...,negative


In [8]:
df['sentiment'].value_counts()

sentiment
positive    250
negative    250
Name: count, dtype: int64

In [9]:
x = df['sentiment'].isin(['positive','negative'])
df = df[x]

In [10]:
df['sentiment'] = df['sentiment'].map({'positive':1, 'negative':0})
df.head()

,review,sentiment
489,main reason see film warren william top form s...,1
255,well much really say really good good job soft...,1
52,excellent example early bette davis talent pro...,1
129,art house maven dream overrated overpraised ov...,0
818,poor film must give lead actress one francine ...,0


In [11]:
df.isnull().sum()

review       0
sentiment    0
dtype: int64

In [ ]:
vectorizer = CountVectorizer(max_features=50)
X = vectorizer.fit_transform(df['review'])
y = df['sentiment']

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.20, random_state=42)

In [16]:
import dagshub

mlflow.set_tracking_uri('https://dagshub.com/sharadkrishna510/Capstone-MLOps-CICD-EKS-Prometheus-Grafana-Project.mlflow')
dagshub.init(repo_owner='sharadkrishna510', repo_name='Capstone-MLOps-CICD-EKS-Prometheus-Grafana-Project', mlflow=True)

# mlflow.set_experiment("Logistic Regression Baseline")
mlflow.set_experiment("Logistic Regression Baseline")


❗❗❗ AUTHORIZATION REQUIRED ❗❗❗



Open the following link in your browser to authorize the client:
https://dagshub.com/login/oauth/authorize?state=5fc132aa-eb4b-4616-98ce-257966923584&client_id=32b60ba385aa7cecf24046d8195a71c07dd345d9657977863b52e7748e0f0f28&middleman_request_id=71634bfc5d2e3aa48ecd11175326bb5cb2d7f7d5568810570e57341c48a1a36b




Accessing as sharadkrishna510

Initialized MLflow to track repo "sharadkrishna510/Capstone-MLOps-CICD-EKS-Prometheus-Grafana-Project"

Repository sharadkrishna510/Capstone-MLOps-CICD-EKS-Prometheus-Grafana-Project initialized!

2026/06/18 15:40:50 INFO mlflow.tracking.fluent: Experiment with name 'Logistic Regression Baseline' does not exist. Creating a new experiment.


<Experiment: artifact_location='mlflow-artifacts:/064d2b09cf904513a7f3e96fda64fab7', creation_time=1781777451030, effective_trace_archival_retention=None, experiment_id='0', last_update_time=1781777451030, lifecycle_stage='active', name='Logistic Regression Baseline', tags={}, trace_location=None, workspace='default'>

In [18]:
import mlflow
import logging
import os
import time
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

# Configure logging
logging.basicConfig(level=logging.INFO, format="%(asctime)s - %(levelname)s - %(message)s")

logging.info("Starting MLflow run...")

with mlflow.start_run():
    start_time = time.time()
    
    try:
        logging.info("Logging preprocessing parameters...")
        mlflow.log_param("vectorizer", "Bag of Words")
        mlflow.log_param("num_features", 50)
        mlflow.log_param("test_size", 0.20)

        logging.info("Initializing Logistic Regression model...")
        model = LogisticRegression(max_iter=1000)  # Increase max_iter to prevent non-convergence issues

        logging.info("Fitting the model...")
        model.fit(X_train, y_train)
        logging.info("Model training complete.")

        logging.info("Logging model parameters...")
        mlflow.log_param("model", "Logistic Regression")

        logging.info("Making predictions...")
        y_pred = model.predict(X_test)

        logging.info("Calculating evaluation metrics...")
        accuracy = accuracy_score(y_test, y_pred)
        precision = precision_score(y_test, y_pred)
        recall = recall_score(y_test, y_pred)
        f1 = f1_score(y_test, y_pred)

        logging.info("Logging evaluation metrics...")
        mlflow.log_metric("accuracy", accuracy)
        mlflow.log_metric("precision", precision)
        mlflow.log_metric("recall", recall)
        mlflow.log_metric("f1_score", f1)

        logging.info("Saving and logging the model...")
        mlflow.sklearn.log_model(model, "model")

        # Log execution time
        end_time = time.time()
        logging.info(f"Model training and logging completed in {end_time - start_time:.2f} seconds.")

        # Save and log the notebook
        # notebook_path = "exp1_baseline_model.ipynb"
        # logging.info("Executing Jupyter Notebook. This may take a while...")
        # os.system(f"jupyter nbconvert --to notebook --execute --inplace {notebook_path}")
        # mlflow.log_artifact(notebook_path)

        # logging.info("Notebook execution and logging complete.")

        # Print the results for verification
        logging.info(f"Accuracy: {accuracy}")
        logging.info(f"Precision: {precision}")
        logging.info(f"Recall: {recall}")
        logging.info(f"F1 Score: {f1}")

    except Exception as e:
        logging.error(f"An error occurred: {e}", exc_info=True)


2026-06-18 16:34:06,800 - INFO - Starting MLflow run...
2026-06-18 16:34:10,334 - INFO - Logging preprocessing parameters...
2026-06-18 16:34:13,248 - INFO - Initializing Logistic Regression model...
2026-06-18 16:34:13,250 - INFO - Fitting the model...
2026-06-18 16:34:13,265 - INFO - Model training complete.
2026-06-18 16:34:13,266 - INFO - Logging model parameters...
2026-06-18 16:34:13,738 - INFO - Making predictions...
2026-06-18 16:34:13,739 - INFO - Calculating evaluation metrics...
2026-06-18 16:34:13,744 - INFO - Logging evaluation metrics...
2026-06-18 16:34:16,412 - INFO - Saving and logging the model...
2026/06/18 16:34:16 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/06/18 16:34:20 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The rec

🏃 View run shivering-hare-928 at: https://dagshub.com/sharadkrishna510/Capstone-MLOps-CICD-EKS-Prometheus-Grafana-Project.mlflow/#/experiments/0/runs/5400c66dd1fb45f0afeec94f3625d5a6
🧪 View experiment at: https://dagshub.com/sharadkrishna510/Capstone-MLOps-CICD-EKS-Prometheus-Grafana-Project.mlflow/#/experiments/0
